Middleware 


In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summarization middleware


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage , SystemMessage

### message summarization 

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            model_provider="groq",
            trigger=("messages",10),
            keep=("messages",4),
        )
    ]    

)



In [6]:
### Run with thread id

config = {"configurable":{"thread_id":"test-1"}}

In [7]:
# alternative test data 
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='1f617070-07a8-422d-b19a-6164670bf23e'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. We just answer.'}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 78, 'total_tokens': 110, 'completion_time': 0.070482716, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.003455955, 'prompt_tokens_details': None, 'queue_time': 0.369773894, 'total_time': 0.073938671}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a01a83-3bdb-72a2-bfc2-f1ef21ab0565-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 32, 'total_tokens': 110, 'output_token_details': {'reasoning': 13}})]}
Messages

### Token Size

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_hotel(city: str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 Star, $180/night, Business center
    """

# 1. Define the custom token counter BEFORE using it
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    # 2. Pass tools here!
    tools=[search_hotel], 
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            # 3. Use plural "tokens"
            trigger=("tokens", 100), 
            # 4. Keep fewer messages so it actually summarizes older ones
            keep=("messages", 4), 
            # 5. Connect your custom token counter
            token_counter=count_token 
        )
    ]    
)

config = {"configurable":{"thread_id":"test-1"}}

# 1. Ask the agent a question that requires using the tool
inputs = {"messages": [("user", "Can you search for a hotel in Tokyo?")]}

# 2. Invoke the agent
response = agent.invoke(inputs, config=config)

# 3. Print the final response
print(response["messages"][-1].content)



Here are a couple of options to get you started:

| # | Hotel | Rating | Approx. Price/night | Key Amenities |
|---|-------|--------|---------------------|----------------|
| 1 | **Grand Hotel** | 5 ★ | **$350** | Spa, outdoor pool, fully‑equipped gym, rooftop bar, concierge services |
| 2 | **City Inn** | 4 ★ | **$180** | Business center, free Wi‑Fi, on‑site restaurant, close to major subway lines |

Both are centrally located in Tokyo and have good guest reviews. Let me know if any of these look promising or if you have specific preferences (e.g., budget range, location near a particular district, family‑friendly, pet‑friendly, etc.) and I can refine the search further!
